In [1]:
import numpy as np
from astropy.io import fits

In [8]:
opencov = fits.open('../data/covariance/xip_xim_map3_covariance_8ARCMINCUT_31Oct24.fits')
cov_matrix = opencov[1].data
opendata_2pt = fits.open('../data/dv/sim_2pt-NLA-cosmoCosmogrid-04Nov24.fits')
opendata_map3 = fits.open('../data/dv/sim_map3-cosmoCosmogrid-6Nov24-8arcmincut-WITH_COV.fits')

data_vector = np.zeros(480)
for i in range(480):
    if i < 200:
        data_vector[i] = opendata_2pt[2].data[i][3]
    elif i < 400:
        data_vector[i] = opendata_2pt[3].data[i-200][3]
    else:
        data_vector[i] = opendata_map3[1].data[i-400][6]

In [35]:
# Parameters
n_noisy_samples = 1000000  # Number of noisy samples to generate

# Generate 2000 noisy data vectors
noisy_data_vectors = np.random.multivariate_normal(mean=data_vector, cov=cov_matrix, size=n_noisy_samples)

# Compute the empirical covariance matrix of the generated noisy data vectors
empirical_cov_matrix = np.cov(noisy_data_vectors, rowvar=False)

# Calculate the normalized Frobenius norm of the difference between covariances
frobenius_diff = np.linalg.norm(empirical_cov_matrix - cov_matrix, ord='fro') / np.linalg.norm(cov_matrix, ord='fro')

# Output the results
print("Normalized Frobenius difference between empirical and original covariance matrices:", frobenius_diff)

# Optional: Threshold for similarity (you may adjust this threshold)
threshold = 0.0  # Example threshold for similarity (10% difference)
if frobenius_diff < threshold:
    print("The empirical covariance matrix is similar to the original covariance matrix.")
else:
    print("The empirical covariance matrix is significantly different from the original covariance matrix.")

Normalized Frobenius difference between empirical and original covariance matrices: 0.006265190209463862
The empirical covariance matrix is similar to the original covariance matrix.


In [42]:
#Generate the first noisy realization

noisy_data_vector = np.random.multivariate_normal(mean=data_vector, cov=cov_matrix, size=10)
print(np.shape(noisy_data_vector))

(10, 480)


In [43]:
def save_noisy_realization(number, noisy_dvs):
    opendata_2pt = fits.open('../data/dv/sim_2pt-NLA-cosmoCosmogrid-04Nov24.fits')
    opendata_2pt_500simcov = fits.open('../data/dv/sim_2pt-NLA-cosmoCosmogrid-04Nov24_500simcov.fits')
    opendata_map3 = fits.open('../data/dv/sim_map3-cosmoCosmogrid-6Nov24-8arcmincut-WITH_COV.fits')
 
    data_vector = np.zeros(480)
    for i in range(480):
        if i < 200:
            opendata_2pt[2].data[i][3] = noisy_data_vector[number][i]
            opendata_2pt_500simcov[2].data[i][3] = noisy_data_vector[number][i]
        elif i < 400:
            opendata_2pt[3].data[i-200][3] = noisy_data_vector[number][i]
            opendata_2pt_500simcov[3].data[i-200][3] = noisy_data_vector[number][i]
        else:
            opendata_map3[1].data[i-400][6] = noisy_data_vector[number][i]
        
    noisy_2pt = '/Users/gchgomes/3pcf_integrator/data/dv/noisy_realizations/sim_2pt-NLA-cosmoCosmogrid-anacov-00'+str(number+1)+'.fits'
    opendata_2pt.writeto(noisy_2pt)   

    noisy_2pt_500simcov = '/Users/gchgomes/3pcf_integrator/data/dv/noisy_realizations/sim_2pt-NLA-cosmoCosmogrid-500simcov-00'+str(number+1)+'.fits'
    opendata_2pt_500simcov.writeto(noisy_2pt_500simcov) 

    noisy_map3 = '/Users/gchgomes/3pcf_integrator/data/dv/noisy_realizations/sim_map3-NLA-cosmoCosmogrid-00'+str(number+1)+'.fits'
    opendata_map3.writeto(noisy_map3) 

In [63]:
#Now we geneare noisy dv with the compressed 2pt function

transform = np.loadtxt("../data/moped/moped-moped-compress-2pt.txt")
dv2pt = data_vector[:400]
scale_cuts_xip = np.array([20,21,22,23,40,41,42,43,60,61,62,80,81,82,83,100,101,102,103,120,121,122,123,140,141,142,143,160,161,162,163,180,181,182])
scale_cuts_xim = np.concatenate((np.arange(200,210), np.arange(220,234),np.arange(240,254), np.arange(260,273), np.arange(280,294), np.arange(300,315), np.arange(320,335), np.arange(340,355), np.arange(360,375), np.arange(380,394)))
scale_cuts = np.concatenate((scale_cuts_xip, scale_cuts_xim))
scalecut_dv2pt = np.delete(dv2pt, scale_cuts)
compressed_dv = np.zeros(96)
compressed_dv[:16] = np.dot(transform.T, scalecut_dv2pt)
compressed_dv[16:] = data_vector[400:]

In [64]:
#Now we generate the transformed cov

transform_joint = np.zeros((307,96))
transform_joint[:227,:16] = transform
transform_joint[227:,16:] = np.eye(80)

cov_cut = np.delete(np.delete(cov_matrix, scale_cuts, axis=0), scale_cuts, axis=1)
compressed_cov = np.dot(transform_joint.T, np.dot(cov_cut, transform_joint))

noisy_data_vectors = np.random.multivariate_normal(mean=compressed_dv, cov=compressed_cov, size=10)
np.shape(noisy_data_vectors)

(10, 96)

In [70]:
def save_noisy_compressed_realization(number, noisy_dvs):

    opendata_map3 = fits.open('../data/dv/sim_map3-cosmoCosmogrid-6Nov24-8arcmincut-WITH_COV.fits')
 
    cut_compressed_2pt = noisy_data_vector[number][:16]
    for i in range(16,96):
        opendata_map3[1].data[i-16][6] = noisy_data_vector[number][i]

    noisy_map3 = '../data/dv/noisy_realizations/sim_map3-NLA-cosmoCosmogrid-00'+str(number+1)+'.fits'
    opendata_map3.writeto(noisy_map3) 
    
    np.savetxt('../data/dv/compressed_moped/sim-2pt-noisy-compressed-00'+str(number+1)+'.txt',cut_compressed_2pt)

In [71]:
for j in range(9):
    save_noisy_compressed_realization(j,noisy_data_vectors)